In [4]:
# Lab 4 — Reflex Agents (Vacuum Cleaner, Smart Light, Thermostat)
#This notebook implements:
#1. 3-room vacuum cleaner agent (with random move + cleaning counter)
#2. Rule-based Smart Light agent
#3. Thermostat agent with memory (model-based)


In [6]:
import random

class Environment:
    def __init__(self):
        self.rooms = {'A': 'Dirty', 'B': 'Dirty', 'C': 'Dirty'}
        self.agent_location = 'A'
    def get_percept(self):
        return self.agent_location, self.rooms[self.agent_location]
    def execute_action(self, action):
        name = action[0]
        if name == 'Vacuum':
            self.rooms[self.agent_location] = 'Clean'
        elif name == 'MoveTo':
            target = action[1]
            if target in self.rooms:
                self.agent_location = target
        elif name == 'NoOp':
            pass
    def all_clean(self):
        return all(status == 'Clean' for status in self.rooms.values())

class SimpleReflexAgent:
    def __init__(self):
        self.rules = {
            ('A', 'Dirty'): ('Vacuum',),
            ('B', 'Dirty'): ('Vacuum',),
            ('C', 'Dirty'): ('Vacuum',),
        }
        self.clean_count = 0
    def select_action(self, percept, env):
        if percept in self.rules:
            action = self.rules[percept]
            if action[0] == 'Vacuum':
                self.clean_count += 1
            return action
        if env.all_clean():
            possible_rooms = list(env.rooms.keys())
            possible_rooms.remove(percept[0])
            target = random.choice(possible_rooms)
            return ('MoveTo', target)
        loc = percept[0]
        if loc == 'A': return ('MoveTo', 'B')
        if loc == 'B': return ('MoveTo', 'C')
        return ('MoveTo', 'A')

def run_simulation(steps=10, seed=None):
    if seed is not None: random.seed(seed)
    env = Environment()
    agent = SimpleReflexAgent()
    for step in range(steps):
        percept = env.get_percept()
        action = agent.select_action(percept, env)
        print(f"Step {step+1}: Percept={percept}, Action={action}, Rooms={env.rooms}")
        env.execute_action(action)
    print(f"\nTotal times vacuumed: {agent.clean_count}")
    print(f"Final state: Agent at {env.agent_location}, Rooms: {env.rooms}")

run_simulation(steps=12, seed=42)


Step 1: Percept=('A', 'Dirty'), Action=('Vacuum',), Rooms={'A': 'Dirty', 'B': 'Dirty', 'C': 'Dirty'}
Step 2: Percept=('A', 'Clean'), Action=('MoveTo', 'B'), Rooms={'A': 'Clean', 'B': 'Dirty', 'C': 'Dirty'}
Step 3: Percept=('B', 'Dirty'), Action=('Vacuum',), Rooms={'A': 'Clean', 'B': 'Dirty', 'C': 'Dirty'}
Step 4: Percept=('B', 'Clean'), Action=('MoveTo', 'C'), Rooms={'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'}
Step 5: Percept=('C', 'Dirty'), Action=('Vacuum',), Rooms={'A': 'Clean', 'B': 'Clean', 'C': 'Dirty'}
Step 6: Percept=('C', 'Clean'), Action=('MoveTo', 'A'), Rooms={'A': 'Clean', 'B': 'Clean', 'C': 'Clean'}
Step 7: Percept=('A', 'Clean'), Action=('MoveTo', 'B'), Rooms={'A': 'Clean', 'B': 'Clean', 'C': 'Clean'}
Step 8: Percept=('B', 'Clean'), Action=('MoveTo', 'C'), Rooms={'A': 'Clean', 'B': 'Clean', 'C': 'Clean'}
Step 9: Percept=('C', 'Clean'), Action=('MoveTo', 'A'), Rooms={'A': 'Clean', 'B': 'Clean', 'C': 'Clean'}
Step 10: Percept=('A', 'Clean'), Action=('MoveTo', 'B'), Rooms={'A'

In [7]:
import time

class LightEnvironment:
    def __init__(self):
        self.rooms = {'Office': {'light_state': 'Off', 'ambient': 'Dim', 'motion': False}}
    def get_percept(self, room):
        r = self.rooms[room]
        return (room, r['motion'], r['ambient'])
    def set_sensor(self, room, motion=None, ambient=None):
        if motion is not None:
            self.rooms[room]['motion'] = motion
        if ambient is not None:
            self.rooms[room]['ambient'] = ambient
    def execute_action(self, room, action):
        self.rooms[room]['light_state'] = action

class SmartLightAgent:
    def __init__(self, timeout=5):
        self.last_motion = {}
        self.timeout = timeout
    def select_action(self, percept, now):
        room, motion, ambient = percept
        if motion:
            self.last_motion[room] = now
            if ambient == 'Dim':
                return 'TurnOn'
            return 'NoOp'
        last = self.last_motion.get(room, None)
        if last and (now - last) > self.timeout:
            return 'TurnOff'
        return 'NoOp'

# test
env = LightEnvironment()
agent = SmartLightAgent(timeout=3)
env.set_sensor('Office', motion=True, ambient='Dim')
print("Action:", agent.select_action(env.get_percept('Office'), time.time()))
time.sleep(4)
env.set_sensor('Office', motion=False)
print("Action after 4s:", agent.select_action(env.get_percept('Office'), time.time()))


Action: TurnOn
Action after 4s: TurnOff


In [8]:
from collections import deque

class ThermostatAgent:
    def __init__(self, history_size=5, low=20, high=24, slope_threshold=0.5):
        self.history = deque(maxlen=history_size)
        self.low, self.high = low, high
        self.slope_threshold = slope_threshold
        self.mode = 'Idle'
    def sense(self, temp):
        self.history.append(float(temp))
    def compute_slope(self):
        if len(self.history) < 2:
            return 0.0
        return (self.history[-1] - self.history[0]) / (len(self.history)-1)
    def select_action(self):
        if not self.history:
            return 'NoOp'
        t = self.history[-1]
        slope = self.compute_slope()
        if t < self.low:
            self.mode = 'Heating'; return 'Heat'
        if t > self.high:
            self.mode = 'Cooling'; return 'Cool'
        if slope > self.slope_threshold and t > (self.high-1):
            self.mode = 'Cooling'; return 'PreCool'
        if slope < -self.slope_threshold and t < (self.low+1):
            self.mode = 'Heating'; return 'PreHeat'
        self.mode = 'Idle'; return 'NoOp'

def simulate_temps(temps):
    agent = ThermostatAgent()
    for i, t in enumerate(temps):
        agent.sense(t)
        print(f"Step {i+1}: Temp={t:.1f}°C, Slope={agent.compute_slope():.2f}, Action={agent.select_action()} ({agent.mode})")

simulate_temps([22.0, 23.0, 24.5, 25.0, 25.6])


Step 1: Temp=22.0°C, Slope=0.00, Action=NoOp (Idle)
Step 2: Temp=23.0°C, Slope=1.00, Action=NoOp (Idle)
Step 3: Temp=24.5°C, Slope=1.25, Action=Cool (Cooling)
Step 4: Temp=25.0°C, Slope=1.00, Action=Cool (Cooling)
Step 5: Temp=25.6°C, Slope=0.90, Action=Cool (Cooling)
